the models i will be comparing here are:  

Decision Trees  
Random Forests  
K-Nearest Neighbors (KNN)  
gradient boosting  
logistic regression

In [16]:
import pandas as pd 
import numpy as np

# models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# grid search
from sklearn.model_selection import GridSearchCV

# metrics (evaluation)
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, confusion_matrix, accuracy_score, f1_score

In [2]:
X_train = pd.read_csv("../data/clean_data/X_train_final.csv")
y_train = pd.read_csv("../data/clean_data/y_train_final.csv")
X_val = pd.read_csv("../data/clean_data/X_val_final.csv")
y_val = pd.read_csv("../data/clean_data/y_val_final.csv")

In [3]:
y_train = y_train.to_numpy()
y_train = y_train.reshape(-1,1)
y_val = y_val.to_numpy()
y_val = y_val.reshape(-1,1)

### base model loop

In [4]:
# initializing base models
models = {
    "logistic_regression" : LogisticRegression(max_iter=1000, random_state=42),
    "random_forest" : RandomForestClassifier(random_state=42),
    "KNN" : KNeighborsClassifier(),
    "decision_tree" : DecisionTreeClassifier(random_state=42),
    "XGB" : XGBClassifier(random_state=42, eval_metric='logloss')
}

In [5]:
results = []

for name, model in models.items():

    print(f"model: {models[name]}")

    model.fit(X_train, y_train.ravel())

    y_pred_val = model.predict(X_val)
    y_pred_train = model.predict(X_train)

    # model evaluation
    results.append({
        "model" : name,
        "accuracy_train" : accuracy_score(y_train.ravel(), y_pred_train),
        "accuracy_val" : accuracy_score(y_val.ravel(), y_pred_val),
        "precision_train" : precision_score(y_train.ravel(), y_pred_train, zero_division=0),
        "precision_val" : precision_score(y_val.ravel(), y_pred_val, zero_division=0),
        "recall_train" : recall_score(y_train.ravel(), y_pred_train, zero_division=0),
        "recall_val" : recall_score(y_val.ravel(), y_pred_val, zero_division=0),
        "f1_train" : f1_score(y_train.ravel(), y_pred_train,zero_division=0),
        "f1_val" : f1_score(y_val.ravel(), y_pred_val,zero_division=0),
    })

df_results = pd.DataFrame(results).sort_values(by="recall_val", ascending=False)
display(df_results)

model: LogisticRegression(max_iter=1000, random_state=42)
model: RandomForestClassifier(random_state=42)
model: KNeighborsClassifier()
model: DecisionTreeClassifier(random_state=42)
model: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)


,model,accuracy_train,accuracy_val,precision_train,precision_val,recall_train,recall_val,f1_train,f1_val
4,XGB,1.000000,0.848054,1.0,0.005479,1.000000,1.0,1.000000,0.010899
3,decision_tree,1.000000,0.841356,1.0,0.005249,1.000000,1.0,1.000000,0.010444
0,logistic_regression,1.000000,0.778987,1.0,0.001894,1.000000,0.5,1.000000,0.003774
2,KNN,0.777778,0.913353,1.0,0.004831,0.555556,0.5,0.714286,0.009569
1,random_forest,1.000000,0.712013,1.0,0.001453,1.000000,0.5,1.000000,0.002899


having a look at the resulting dataframe almost all the models score perfect results for the training data yet 0 for the validation data.   
that shows a severe overfitting and an insane data imbalance as expected.  

### coarse grid search 

In [8]:
# initializing base models
models_coarse = {
    "logistic_regression" : LogisticRegression(max_iter=1000, random_state=42),
    "random_forest" : RandomForestClassifier(random_state=42),
    "KNN" : KNeighborsClassifier(),
    "decision_tree" : DecisionTreeClassifier(random_state=42),
    "XGB" : XGBClassifier(random_state=42, eval_metric='logloss')
}

In [9]:
param_grid_coarse = {
    "logistic_regression" : {
        "C" : [0.01, 0.1, 1.0, 10.0],                                       # inverse of regularization 
    },
    "random_forest" : {
        "n_estimators" : [100, 200, 300],                                   # number of trees created to learn the data
        "class_weight" : ["balanced", "balanced_subsample", None],          
        "min_samples_split" : [2, 5],                                       # minimum number of samples required to split a node (constrains the model to prevent overfitting)
        "max_depth" : [3, 5, 10]                                            # the maximum depth of each tree  (controlling the model complexity to prevent overfitting)
    },
    "KNN" : {
        'n_neighbors': [3, 5, 7, 9, 11],                                    # number of kneighboring samples to be considered
        'weights': ['uniform', 'distance'],                                 # how kneighboring samples influence the model ("uniform": all samples have the same influence, "distance": the closer the higher the influence)
        'metric': ['euclidean', 'manhattan']                                # measures similarity between points.
    },
    "decision_tree" : {
        "max_depth" : [10, 20, 30],                                   
        "min_samples_leaf" : [1, 2, 4],                                     # the minimum number of samples required to create a node (leaf)
        "min_samples_split" : [2, 5, 10],
        "criterion" : ["gini", "entropy"]                                   # determines how best split is chosen ("gini": gini impurity, "entropy": information gain)
    },
    "XGB" : {
        'n_estimators' : [100, 200, 300],                                    
        'learning_rate' : [0.01, 0.05, 0.1, 0.2],                           # how big the steps the model takes towards the global minima
        'max_depth' : [3, 6, 9],
        'subsample' : [0.8, 1.0],                                           # Setting it to 0.5 means that XGBoost would randomly sample half of the training data prior to growing trees.
    }
}

those params are chosen as a fix for the problems we diagnosed earlier. each data set requires different solution hence different params.

In [10]:
results_coarse = []
best_params_coarse = {}

# creating the coarse grid loop
for name in models_coarse.keys():

    cores = 1 if name == "XGB" else -1

    print(f"model: {models_coarse[name]}")

    G_search = GridSearchCV(
        estimator= models_coarse[name],
        param_grid= param_grid_coarse[name],
        n_jobs= cores,
        scoring="f1",
        cv=5,
        verbose=1
    )

    G_search.fit(X_train, y_train.ravel())

    # predicting both the training and validation output
    y_pred_val_coarse = G_search.predict(X_val)
    y_pred_train_coarse = G_search.predict(X_train)

    # evaluation cheat code
    results_coarse.append({
        "model" : name,
        "accuracy_train" : accuracy_score(y_train.ravel(), y_pred_train_coarse),
        "accuracy_val" : accuracy_score(y_val.ravel(), y_pred_val_coarse),
        "precision_train" : precision_score(y_train.ravel(), y_pred_train_coarse, zero_division=0),
        "precision_val" : precision_score(y_val.ravel(), y_pred_val_coarse, zero_division=0),
        "recall_train" : recall_score(y_train.ravel(), y_pred_train_coarse, zero_division=0),
        "recall_val" : recall_score(y_val.ravel(), y_pred_val_coarse, zero_division=0),
        "f1_train" : f1_score(y_train.ravel(), y_pred_train_coarse,zero_division=0),
        "f1_val" : f1_score(y_val.ravel(), y_pred_val_coarse,zero_division=0),
        "best_params_coarse" : str(G_search.best_params_)
    })

    print(f"Confusion Matrix")
    print(confusion_matrix(y_val, y_pred_val_coarse.ravel()))
    print("-" * 35 + "\n")

df_results_coarse = pd.DataFrame(results_coarse).sort_values(by="f1_val", ascending=False)
display(df_results_coarse)

model: LogisticRegression(max_iter=1000, random_state=42)
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Confusion Matrix
[[1733  654]
 [   1    1]]
-----------------------------------

model: RandomForestClassifier(random_state=42)
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Confusion Matrix
[[1696  691]
 [   1    1]]
-----------------------------------

model: KNeighborsClassifier()
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Confusion Matrix
[[2166  221]
 [   1    1]]
-----------------------------------

model: DecisionTreeClassifier(random_state=42)
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Confusion Matrix
[[2008  379]
 [   0    2]]
-----------------------------------

model: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, e

,model,accuracy_train,accuracy_val,precision_train,precision_val,recall_train,recall_val,f1_train,f1_val,best_params_coarse
4,XGB,1.000000,0.860193,1.000000,0.005952,1.000000,1.0,1.00,0.011834,"{'learning_rate': 0.01, 'max_depth': 3, 'n_est..."
3,decision_tree,1.000000,0.841356,1.000000,0.005249,1.000000,1.0,1.00,0.010444,"{'criterion': 'gini', 'max_depth': 10, 'min_sa..."
2,KNN,1.000000,0.907074,1.000000,0.004505,1.000000,0.5,1.00,0.008929,"{'metric': 'manhattan', 'n_neighbors': 9, 'wei..."
0,logistic_regression,0.777778,0.725827,0.857143,0.001527,0.666667,0.5,0.75,0.003044,{'C': 0.01}
1,random_forest,1.000000,0.710339,1.000000,0.001445,1.000000,0.5,1.00,0.002882,"{'class_weight': 'balanced', 'max_depth': 3, '..."


the results after the coarse grid still disappointed me as all the models got 0 on the recall, precision and f1 score which means there is still a severe overfitting. the cause might be the smote due to the way it works so we will try to swap it with randomundersampler.  
the excpected results are a lot better results in the training and validation tests.    
we will try to get the most of these models using the fine grid

In [11]:
df_results_coarse[['model', 'best_params_coarse']]

,model,best_params_coarse
4,XGB,"{'learning_rate': 0.01, 'max_depth': 3, 'n_est..."
3,decision_tree,"{'criterion': 'gini', 'max_depth': 10, 'min_sa..."
2,KNN,"{'metric': 'manhattan', 'n_neighbors': 9, 'wei..."
0,logistic_regression,{'C': 0.01}
1,random_forest,"{'class_weight': 'balanced', 'max_depth': 3, '..."


### fine grid

In [12]:
# initializing base models
models_fine = {
    "logistic_regression" : LogisticRegression(max_iter=1000, random_state=42),
    "random_forest" : RandomForestClassifier(random_state=42),
    "KNN" : KNeighborsClassifier(),
    "decision_tree" : DecisionTreeClassifier(random_state=42),
    "XGB" : XGBClassifier(random_state=42, eval_metric='logloss')
}

In [13]:
param_grid_fine = {
    "logistic_regression" : {
        "C" : [0.001, 0.005, 0.01, 0.015],                                       
    }, 
    "random_forest" : {
        "n_estimators" : [80, 100, 120],                                   
        "class_weight" : ["balanced"],          
        "min_samples_split" : [4, 5, 6],                                       
        "max_depth" : [2, 3, 4]                                            
    },
    "KNN" : {
        'n_neighbors': [8, 9, 10],                                    
        'weights': ['distance'],                                 
        'metric': ['manhattan']                                
    },
    "decision_tree" : {
        "max_depth" : [8, 10, 12],                                   
        "min_samples_leaf" : [1, 2],                                     
        "min_samples_split" : [2, 3, 4],
        "criterion" : ["gini"]                                   
    },
    "XGB" : {
        'n_estimators' : [180, 200, 220],                                    
        'learning_rate' : [0.005, 0.01, 0.015],                           
        'max_depth' : [2, 3, 4],
        'subsample' : [1.0],                                           
    }
}

In [14]:
results_fine = []
best_params_fine = {}

# creating the fine grid loop
for name in models_fine.keys():

    cores = 1 if name == "XGB" else -1

    print(f"model: {models_fine[name]}")

    G_search = GridSearchCV(
        estimator= models_fine[name],
        param_grid= param_grid_fine[name],
        n_jobs= cores,
        scoring="f1",
        cv=5,
        verbose=1
    )

    G_search.fit(X_train, y_train.ravel())

    # predicting both the training and validation output
    y_pred_val_fine = G_search.predict(X_val)
    y_pred_train_fine = G_search.predict(X_train)

    # evaluation cheat code
    results_fine.append({
        "model" : name,
        "accuracy_train" : accuracy_score(y_train.ravel(), y_pred_train_fine),
        "accuracy_val" : accuracy_score(y_val.ravel(), y_pred_val_fine),
        "precision_train" : precision_score(y_train.ravel(), y_pred_train_fine, zero_division=0),
        "precision_val" : precision_score(y_val.ravel(), y_pred_val_fine, zero_division=0),
        "recall_train" : recall_score(y_train.ravel(), y_pred_train_fine, zero_division=0),
        "recall_val" : recall_score(y_val.ravel(), y_pred_val_fine, zero_division=0),
        "f1_train" : f1_score(y_train.ravel(), y_pred_train_fine,zero_division=0),
        "f1_val" : f1_score(y_val.ravel(), y_pred_val_fine,zero_division=0),
        "best_params_fine" : str(G_search.best_params_)
    })

    print(f"Confusion Matrix")
    print(confusion_matrix(y_val, y_pred_val_fine.ravel()))
    print("-" * 35 + "\n")

df_results_fine = pd.DataFrame(results_fine).sort_values(by="f1_val", ascending=False)
display(df_results_fine)

model: LogisticRegression(max_iter=1000, random_state=42)
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Confusion Matrix
[[1700  687]
 [   1    1]]
-----------------------------------

model: RandomForestClassifier(random_state=42)
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Confusion Matrix
[[1724  663]
 [   1    1]]
-----------------------------------

model: KNeighborsClassifier()
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Confusion Matrix
[[2166  221]
 [   1    1]]
-----------------------------------

model: DecisionTreeClassifier(random_state=42)
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Confusion Matrix
[[2008  379]
 [   0    2]]
-----------------------------------

model: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval

,model,accuracy_train,accuracy_val,precision_train,precision_val,recall_train,recall_val,f1_train,f1_val,best_params_fine
4,XGB,1.000000,0.857262,1.000000,0.005831,1.000000,1.0,1.00,0.011594,"{'learning_rate': 0.01, 'max_depth': 2, 'n_est..."
3,decision_tree,1.000000,0.841356,1.000000,0.005249,1.000000,1.0,1.00,0.010444,"{'criterion': 'gini', 'max_depth': 8, 'min_sam..."
2,KNN,1.000000,0.907074,1.000000,0.004505,1.000000,0.5,1.00,0.008929,"{'metric': 'manhattan', 'n_neighbors': 9, 'wei..."
1,random_forest,1.000000,0.722059,1.000000,0.001506,1.000000,0.5,1.00,0.003003,"{'class_weight': 'balanced', 'max_depth': 2, '..."
0,logistic_regression,0.777778,0.712013,0.857143,0.001453,0.666667,0.5,0.75,0.002899,{'C': 0.005}


In [15]:
display(df_results_fine)
display(df_results_coarse)

,model,accuracy_train,accuracy_val,precision_train,precision_val,recall_train,recall_val,f1_train,f1_val,best_params_fine
4,XGB,1.000000,0.857262,1.000000,0.005831,1.000000,1.0,1.00,0.011594,"{'learning_rate': 0.01, 'max_depth': 2, 'n_est..."
3,decision_tree,1.000000,0.841356,1.000000,0.005249,1.000000,1.0,1.00,0.010444,"{'criterion': 'gini', 'max_depth': 8, 'min_sam..."
2,KNN,1.000000,0.907074,1.000000,0.004505,1.000000,0.5,1.00,0.008929,"{'metric': 'manhattan', 'n_neighbors': 9, 'wei..."
1,random_forest,1.000000,0.722059,1.000000,0.001506,1.000000,0.5,1.00,0.003003,"{'class_weight': 'balanced', 'max_depth': 2, '..."
0,logistic_regression,0.777778,0.712013,0.857143,0.001453,0.666667,0.5,0.75,0.002899,{'C': 0.005}


,model,accuracy_train,accuracy_val,precision_train,precision_val,recall_train,recall_val,f1_train,f1_val,best_params_coarse
4,XGB,1.000000,0.860193,1.000000,0.005952,1.000000,1.0,1.00,0.011834,"{'learning_rate': 0.01, 'max_depth': 3, 'n_est..."
3,decision_tree,1.000000,0.841356,1.000000,0.005249,1.000000,1.0,1.00,0.010444,"{'criterion': 'gini', 'max_depth': 10, 'min_sa..."
2,KNN,1.000000,0.907074,1.000000,0.004505,1.000000,0.5,1.00,0.008929,"{'metric': 'manhattan', 'n_neighbors': 9, 'wei..."
0,logistic_regression,0.777778,0.725827,0.857143,0.001527,0.666667,0.5,0.75,0.003044,{'C': 0.01}
1,random_forest,1.000000,0.710339,1.000000,0.001445,1.000000,0.5,1.00,0.002882,"{'class_weight': 'balanced', 'max_depth': 3, '..."


models usually don't output the classes as binary outputs but as a probability so changing the threshold will probably give better results.  
but we must make sure to consider the trade off between precision and recall.   

In [17]:
xgb_champ = XGBClassifier(
    max_depth=3, 
    learning_rate=0.01, 
    n_estimators=200, 
    subsample=1.0,
    eval_metric='logloss',
    n_jobs=1,
    random_state=42
)

xgb_champ.fit(X_train, y_train.ravel())

probabilities = xgb_champ.predict_proba(X_val)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val, probabilities)

pr_df = pd.DataFrame({
    "Threshold": np.append(thresholds, [1.0]), # Thresholds array is 1 item shorter, so we pad it
    "Precision": precisions,
    "Recall": recalls
})

perfect_recall_df = pr_df[pr_df['Recall'] == 1.0]
optimal_threshold = perfect_recall_df['Threshold'].max()

print(f"The absolute highest threshold we can use without missing a satellite is: {optimal_threshold:.4f}\n")

custom_predictions = (probabilities >= optimal_threshold).astype(int)

print("Custom Threshold Confusion Matrix")
print(confusion_matrix(y_val, custom_predictions))
print(f"Final F1-Score: {f1_score(y_val, custom_predictions, zero_division=0):.4f}")

The absolute highest threshold we can use without missing a satellite is: 0.7366

Custom Threshold Confusion Matrix
[[2247  140]
 [   0    2]]
Final F1-Score: 0.0278


well that is a very very big jump in the false alarms which is good.   
now lets print the final best model 

### best model

In [20]:
xgb_best_model = XGBClassifier(
    max_depth=3, 
    learning_rate=0.01, 
    n_estimators=200, 
    subsample=1.0,
    eval_metric='logloss',
    n_jobs=1,
    random_state=42, 
)

xgb_best_model.fit(X_train, y_train.ravel())

probability = xgb_best_model.predict_proba(X_val)

y_pred_best = (probability[:, 1] >= 0.7366).astype(int)

best_model_results = {
    "Accuracy (Validation)": accuracy_score(y_val, y_pred_best),
    "Precision (Validation)": precision_score(y_val, y_pred_best, zero_division=0),
    "Recall (Validation)": recall_score(y_val, y_pred_best, zero_division=0),
    "F1-Score (Validation)": f1_score(y_val, y_pred_best, zero_division=0)
}

best_model_results_df = pd.DataFrame([best_model_results])
display(best_model_results_df)

,Accuracy (Validation),Precision (Validation),Recall (Validation),F1-Score (Validation)
0,0.941398,0.014085,1.0,0.027778


a better solution is to use anomaly detection algorithms instead as the random under sampler reduces the data so it can decreases the amount of info the model trains on.